# 📈 Backtest Portfolio
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Realosunboy6/free-portfolio-visualizer/blob/main/notebooks/01_backtest_portfolio.ipynb)

Free equivalent of Portfolio Visualizer's **Backtest Portfolio** — unlimited assets, daily data, crypto/international support, contributions/withdrawals, band rebalancing, benchmark comparison, real (inflation-adjusted) results.

Edit the form below, then **Runtime → Run all**.

In [ ]:
#@title Setup — run this first {display-mode: "form"}
try:
    import portlab
except ImportError:
    %pip install -q "portlab @ git+https://github.com/Realosunboy6/free-portfolio-visualizer.git"
    import portlab
print("portlab", portlab.__version__, "ready")

In [ ]:
#@title Portfolio settings {display-mode: "form"}
#@markdown Comma-separated tickers and matching weights (any yfinance symbol works — stocks, ETFs, BTC-USD, international):
#@markdown Pick a famous portfolio (overrides tickers/weights below) or leave as Custom:
lazy_portfolio = "Custom" #@param ["Custom", "Classic 60/40", "Bogleheads Three-Fund", "All Weather (Dalio)", "Golden Butterfly", "Permanent (Browne)", "Ivy 5 (Faber)", "Swensen Yale", "Coffeehouse", "No-Brainer (Bernstein)", "Pinwheel", "Rick Ferri Core Four", "Larry Portfolio", "Couch Potato (Burns)", "Talmud", "Desert"]
tickers = "VTI, TLT, GLD"        #@param {type:"string"}
weights = "60, 30, 10"           #@param {type:"string"}
benchmark = "SPY"                #@param {type:"string"}
start_date = "2010-01-01"        #@param {type:"date"}
end_date = ""                    #@param {type:"string"}
initial_amount = 10000           #@param {type:"number"}
#@markdown Periodic cashflow (+contribution / −withdrawal) and frequency:
cashflow = 0                     #@param {type:"number"}
cashflow_frequency = "monthly"   #@param ["monthly", "quarterly", "annually"]
rebalancing = "annually"         #@param ["never", "monthly", "quarterly", "semiannually", "annually", "bands"]
rebalance_band = 0.05            #@param {type:"number"}
inflation_adjusted_results = True #@param {type:"boolean"}
#@markdown Leverage (PV-style short cash) and taxes (beyond PV — its backtests are pre-tax):
leverage = 1.0                   #@param {type:"number"}
annual_debt_rate = 0.05          #@param {type:"number"}
tax_dividend = 0.0               #@param {type:"number"}
tax_capgains = 0.0               #@param {type:"number"}
SMOKE = False

In [ ]:
from portlab import backtest_portfolio, plots
from portlab.data import get_returns
from portlab.data.macro import get_cpi

if lazy_portfolio != "Custom":
    from portlab.data import get_lazy_portfolio
    alloc = get_lazy_portfolio(lazy_portfolio)
    tick_list = list(alloc)
    print(f"{lazy_portfolio}: {alloc}")
else:
    tick_list = [t.strip().upper() for t in tickers.split(",") if t.strip()]
    alloc = dict(zip(tick_list, [float(x) for x in weights.split(",")]))
end = end_date or None
rets = get_returns(tick_list + [benchmark.strip().upper()], start_date, end)
bench = rets[benchmark.strip().upper()]

cpi = None
if inflation_adjusted_results:
    try:
        cpi = get_cpi(start_date)
    except Exception as e:
        print("CPI unavailable, running nominal:", e)

result = backtest_portfolio(
    rets, alloc, initial=initial_amount, cashflow=cashflow,
    cashflow_freq=cashflow_frequency, rebalance=rebalancing,
    rebalance_band=rebalance_band if rebalancing == "bands" else None,
    benchmark=bench, cpi=cpi,
    leverage=leverage, debt_rate=annual_debt_rate,
    tax_dividend=tax_dividend, tax_capgains=tax_capgains,
    div_yield={t: 0.02 for t in tick_list} if tax_dividend > 0 else None)
if result.margin_calls:
    print("Margin calls on:", [str(d.date()) for d in result.margin_calls])
result.summary().style.format("{:.4f}", na_rep="—")

In [ ]:
import pandas as pd
both = pd.concat([result.returns.rename("Portfolio"), bench.rename("Benchmark")], axis=1)
plots.growth_chart(both, initial=initial_amount).show()
plots.drawdown_chart(both).show()
plots.annual_returns_chart(both).show()

In [ ]:
from portlab import metrics
plots.rolling_chart(metrics.rolling_returns(result.returns, 3).dropna(),
                    "Rolling 3-Year Annualized Return").show()
plots.weights_chart(result.weights, "Allocation Drift").show()
plots.return_contribution_chart(result.weights, rets,
    "Which asset drove the returns?").show()
print("Worst drawdowns:")
result.drawdowns()

In [ ]:
# One-line shareable report (quantstats-style) — downloads in Colab
result.tear_sheet("tear_sheet.html")
try:
    from google.colab import files
    files.download("tear_sheet.html")
except ImportError:
    print("Saved tear_sheet.html")

**Improve on PV:** rerun with `rebalancing='bands'`, add `BTC-USD`, or push the start date back as far as your assets trade — no 15-asset limit, no paywall.